In [1]:
import nltk
from nltk.stem import WordNetLemmatizer
import json
import pickle
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
from tensorflow.keras.optimizers.legacy import SGD
import random

lemmatizer = WordNetLemmatizer()

words=[]
classes = []
documents = []
ignore_words = ['?', '!']
data_file = open('DatasetNew.json').read()
intents = json.loads(data_file)
for intent in intents['intents']:
    for pattern in intent['patterns']:
        #tokenize each word
        w = nltk.word_tokenize(pattern)
        words.extend(w)
        #add documents in the corpus
        documents.append((w, intent['tag']))
        # add to our classes list
        if intent['tag'] not in classes:
            classes.append(intent['tag'])
# lemmaztize and lower each word and remove duplicates
words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))
# sort classes
classes = sorted(list(set(classes)))
# documents = combination between patterns and intents
print(len(documents), "documents")
# classes = intents
print(len(classes), "classes", classes)
# words = all words, vocabulary
print(len(words), "unique lemmatized words", words)

pickle.dump(words, open('texts.pkl', 'wb'))
pickle.dump(classes, open('labels.pkl', 'wb'))

# create our training data
training = []
output_empty = [0] * len(classes)

# training set, bag of words for each sentence
for doc in documents:
    # initialize our bag of words
    bag = []
    # list of tokenized words for the pattern
    pattern_words = doc[0]
    # lemmatize each word - create base word, in an attempt to represent related words
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    # create our bag of words array with 1, if word match found in the current pattern
    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)

    # output is a '0' for each tag and '1' for the current tag (for each pattern)
    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1

    training.append([bag, output_row])

# shuffle our features and turn into np.array
random.shuffle(training)
training = np.array(training, dtype=object)  # Specify dtype=object to allow for variable-length sub-arrays

# create train and test lists. X - patterns, Y - intents
train_x = list(training[:, 0])
train_y = list(training[:, 1])

print("Training data created")

# Create model - 3 layers. First layer 128 neurons, second layer 64 neurons,
# and 3rd output layer contains the number of neurons equal to the number of intents to predict output intent with softmax
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))

# Compile model using updated SGD
sgd = SGD(learning_rate=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

# fitting and saving the model
hist = model.fit(np.array(train_x), np.array(train_y), epochs=200, batch_size=5, verbose=1)
model.save('model.h5')

print("Model created")

/Users/fadhilahmad/anaconda3/envs/fadhil/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


1108 documents
965 classes ['AAUPB', 'Aborsi', 'Acara', 'Acara Pemeriksaan Cepat', 'Aceh', 'Actor Sequitur Forum Rei', 'Adanya Kata Sepakat', 'Adat Bali', 'Adat Batak', 'Adat Jawa', 'Adat Sunda', 'Adat yang Hidup', 'Administrasi', 'Administrasi Negara', 'Adopsi Anak', 'Aduan', 'Advokat', 'Agrikultur', 'Ahli Waris', 'Akta', 'Akta Kematian', 'Akta Otentik', 'Akta di Bawah Tangan', 'Aktor-Aktor Peradilan', 'Akuisisi Perusahaan', 'Akuntan Publik', 'Alasan Pemaaf Pidana', 'Alasan Pembenar Pidana', 'Alat Bukti', 'Amar Putusan', 'Amar Tambahan', 'Amdal', 'Anak Luar Kawin', 'Anak yang Berhadapan dengan Hukum', 'Anak yang Berkonflik dengan Hukum', 'Anak yang Menjadi Korban Tindak Pidana', 'Anak yang Menjadi Saksi Tindak Pidana', 'Analisis Risiko Lingkungan Hidup', 'Anggaran Berbasis Lingkungan Hidup', 'Asal Usul Anak', 'Asas-Asas Hukum', 'Asas-Asas dalam Perjanjian', 'Asuransi', 'Asuransi Syariah', 'Audi Alteram Partem', 'Audit Lingkungan Hidup', 'Badan Pemeriksa Keuangan', 'Badan Penyelesaian 

2024-02-05 19:11:12.750482: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-02-05 19:11:12.750628: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/200


2024-02-05 19:11:13.068776: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2024-02-05 19:11:13.204329: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


222/222 [==============================] - 4s 13ms/step - loss: 6.8800 - accuracy: 0.0045
Epoch 2/200
222/222 [==============================] - 2s 10ms/step - loss: 6.8710 - accuracy: 0.0090
Epoch 3/200
222/222 [==============================] - 2s 10ms/step - loss: 6.8598 - accuracy: 0.0099
Epoch 4/200
222/222 [==============================] - 2s 9ms/step - loss: 6.8468 - accuracy: 0.0099
Epoch 5/200
222/222 [==============================] - 2s 10ms/step - loss: 6.8308 - accuracy: 0.0090
Epoch 6/200
222/222 [==============================] - 2s 10ms/step - loss: 6.8088 - accuracy: 0.0090
Epoch 7/200
222/222 [==============================] - 2s 9ms/step - loss: 6.7625 - accuracy: 0.0090
Epoch 8/200
222/222 [==============================] - 2s 10ms/step - loss: 6.6997 - accuracy: 0.0117
Epoch 9/200
222/222 [==============================] - 2s 10ms/step - loss: 6.6095 - accuracy: 0.0126
Epoch 10/200
222/222 [==============================] - 2s 10ms/step - loss: 6.5587 - accuracy: 